# Chapter 31 bridge: deep networks, initialisation, batch norm

Chapter 31 trains a five-layer network and shows that batch normalization rescues a run that naive initialisation kills.

The check below runs **the chapter's own `train`, read from `c4.py` at run time**, for a single step, and compares the weights it produces against one step of the same update in PyTorch. If the chapter's gradient changes, this check moves with it.

Nothing here is reimplemented. The chapter's own file is executed, its own weights and data are handed to PyTorch, and the chapter's hand-derived gradients are compared against autograd. Tolerances are stated per check and are **relative** to the size of the quantity being compared, because an absolute threshold means nothing without a scale.

Run order: top to bottom, from a fresh kernel. Requires `requirements-bridges.txt` on top of the book's own `requirements.txt`.

In [1]:
import os, sys, numpy as np, torch
torch.set_default_dtype(torch.float64)          # match NumPy's float64 exactly
CH = os.path.join("..", "code", "ch31")
os.chdir(CH) if os.path.basename(os.getcwd()) != "ch31" else None
def run(name):
    exec(open(name, encoding="utf-8").read(), globals())
run("_lib.py")
print("chapter:", os.path.basename(os.getcwd()), "| torch", torch.__version__, "| numpy", np.__version__)


chapter: ch31 | torch 2.14.0 | numpy 2.4.4


In [2]:
def report(name, ours, theirs, tol=1e-9):
    a = np.asarray(ours, dtype=float); b = np.asarray(theirs, dtype=float)
    denom = max(np.abs(b).max(), 1e-300)
    absd = np.abs(a - b).max(); rel = absd / denom
    ok = rel <= tol
    RESULTS.append(dict(check=name, max_abs=float(absd), max_rel=float(rel),
                        scale=float(denom), tol=tol, passed=bool(ok)))
    print(f"{'PASS' if ok else 'FAIL'}  {name:52s} max|diff| {absd:.3e}   "
          f"relative {rel:.2e}   (tolerance {tol:g})")
    return ok
RESULTS = []
MEASUREMENTS = []      # reported, never asserted: these have no single right answer


In [3]:
run("c1.py"); run("c2.py"); run("c3.py"); run("c4.py"); run("c5.py")

  layer   mean |activation|  fraction alive
  input              0.3039              --
      1                 1.8          0.5519
      2                 7.5          0.4585
      3                51.1          0.5363
      4               217.6          0.4732
      5             1,112.0          0.5012
      6             6,585.7          0.4784
      7            39,730.9          0.5072
      8           271,077.1          0.5952
  layer   mean |activation|  fraction alive
  input              0.3039              --
      1              0.3130          0.5519
      2              0.2349          0.4585
      3              0.2821          0.5363
      4              0.2125          0.4732
      5              0.1920          0.5012
      6              0.2010          0.4784
      7              0.2143          0.5072
      8              0.2585          0.5952

why sqrt(2/fan_in): each output is a sum of fan_in terms, each
roughly variance 1 * weight_variance. Setting weight_var

  epoch    0  loss    24.9888  test acc 0.1000
  epoch   50  loss     2.3028  test acc 0.1028
  epoch  100  loss     2.3027  test acc 0.1028
  epoch  150  loss     2.3026  test acc 0.1028
  epoch  200  loss     2.3026  test acc 0.1028
  epoch  250  loss     2.3025  test acc 0.1028

He initialization, same architecture, same learning rate


  epoch    0  loss     2.4655  test acc 0.0972
  epoch   50  loss     0.8202  test acc 0.8194
  epoch  100  loss     0.3258  test acc 0.8889
  epoch  150  loss     0.1886  test acc 0.9167
  epoch  200  loss     0.1278  test acc 0.9444
  epoch  250  loss     0.0999  test acc 0.9583
naive initialization, WITH batch normalization


  epoch    0  loss     8.9811  test acc 0.3083
  epoch   50  loss     1.6946  test acc 0.5833
  epoch  100  loss     1.5755  test acc 0.6194
  epoch  150  loss     1.1951  test acc 0.6389
  epoch  200  loss     0.9892  test acc 0.6500
  epoch  250  loss     0.8389  test acc 0.6806


In [4]:

import re
def one_step(filename, funcname, *a, **kw):
    """Run ONE training step of the chapter's own function and hand back its weights.

    The function's source is READ OUT OF THE CHAPTER FILE at run time, not retyped here and
    not taken from an already-imported object. Edit the chapter's gradient and this check
    moves with it, which is the whole point of a bridge. Only the final `return` is rewritten,
    so the locals -- the updated weights -- come back.
    """
    src = open(filename, encoding='utf-8').read()
    m = re.search(r'^def ' + funcname + r'\(.*?(?=^\S|\Z)', src, re.S | re.M)
    if not m:
        raise SystemExit(funcname + ' not found in ' + filename)
    lines = m.group(0).rstrip().split(chr(10))
    lines[0] = re.sub(r'^def ' + funcname, 'def _instrumented', lines[0])
    for i in range(len(lines) - 1, -1, -1):
        if lines[i].lstrip().startswith('return '):
            pad = lines[i][:len(lines[i]) - len(lines[i].lstrip())]
            lines[i] = pad + 'return locals()'
            break
    g = dict(globals())
    exec(chr(10).join(lines), g)
    print('checking', funcname, 'as read from', filename)
    return g['_instrumented'](*a, **kw)


### One step of the chapter's own `train`, against one step in PyTorch

In [5]:
ETA, SEED = 0.05, 31
# c4.py's train loops `for epoch in range(epochs + 1)`, so epochs=0 is exactly one update.
st = one_step("c4.py", "train", deep_sizes, he_init, seed=SEED, eta=ETA, epochs=0)
Ws_after, bs_after = st["Ws"], st["bs"]

Ws0 = he_init(deep_sizes, seed=SEED); bs0 = [np.zeros(s) for s in deep_sizes[1:]]
tWs = [torch.tensor(w, requires_grad=True) for w in Ws0]
tbs = [torch.tensor(b, requires_grad=True) for b in bs0]
h = torch.tensor(Xtr)
for i, (w, b) in enumerate(zip(tWs, tbs)):
    h = h @ w + b
    if i < len(tWs) - 1: h = torch.relu(h)
tp = torch.softmax(h, dim=1)
tY = torch.tensor(np.eye(10)[ytr])
(-(tY * torch.log(tp + 1e-12)).sum() / len(Xtr)).backward()
for i in range(len(tWs)):
    report(f"one step of train(): W[{i}] after the update",
           Ws_after[i], (tWs[i] - ETA * tWs[i].grad).detach().numpy())
    report(f"one step of train(): b[{i}] after the update",
           bs_after[i], (tbs[i] - ETA * tbs[i].grad).detach().numpy())

checking train as read from c4.py
PASS  one step of train(): W[0] after the update           max|diff| 4.952e-14   relative 7.28e-14   (tolerance 1e-09)
PASS  one step of train(): b[0] after the update           max|diff| 5.739e-14   relative 2.27e-11   (tolerance 1e-09)
PASS  one step of train(): W[1] after the update           max|diff| 1.149e-13   relative 1.28e-13   (tolerance 1e-09)
PASS  one step of train(): b[1] after the update           max|diff| 7.648e-14   relative 2.08e-11   (tolerance 1e-09)
PASS  one step of train(): W[2] after the update           max|diff| 1.835e-13   relative 2.96e-13   (tolerance 1e-09)
PASS  one step of train(): b[2] after the update           max|diff| 1.091e-13   relative 2.40e-11   (tolerance 1e-09)
PASS  one step of train(): W[3] after the update           max|diff| 8.902e-14   relative 1.50e-13   (tolerance 1e-09)
PASS  one step of train(): b[3] after the update           max|diff| 6.930e-14   relative 2.15e-11   (tolerance 1e-09)
PASS  one step

### The batch-norm backward the chapter deliberately does not derive
`c5.py` normalises on the forward pass but propagates the gradient as if the normalization were not there, and says so. This measures the size of that approximation on a **random** incoming gradient. A constant incoming gradient would be useless here: it is batch normalization's null direction, so the true gradient is numerically zero and any ratio against it is meaningless.

This is a measurement, not a pass/fail check, and it is reported as one.

In [6]:
rng = np.random.default_rng(31)
gamma = np.ones(deep_sizes[1]); beta = np.zeros(deep_sizes[1]); eps = 1e-5
W, b = he_init(deep_sizes, seed=31)[0], np.zeros(deep_sizes[1])
Xb = Xtr[:128]
z = Xb @ W + b
mu, var = z.mean(0), z.var(0)
zn = (z - mu) / np.sqrt(var + eps)
out = gamma * zn + beta
dout = rng.normal(0, 1, out.shape)          # a random direction, not the null one

chapter_dz = dout * gamma                   # what c5.py propagates: straight through

tz = torch.tensor(z, requires_grad=True)
tmu = tz.mean(0); tvar = tz.var(0, unbiased=False)
tout = torch.tensor(gamma) * (tz - tmu) / torch.sqrt(tvar + eps) + torch.tensor(beta)
tout.backward(torch.tensor(dout))
true_dz = tz.grad.numpy()

rel = np.abs(chapter_dz - true_dz).max() / np.abs(true_dz).max()
cos = float((chapter_dz * true_dz).sum() /
            (np.linalg.norm(chapter_dz) * np.linalg.norm(true_dz)))
print(f"straight-through dz vs the true batch-norm dz")
print(f"  max relative difference {rel:.3f}")
print(f"  cosine similarity       {cos:.4f}   (1.0 would mean the same direction)")
print("The chapter states this approximation in Steps 5 and 6. It is a real gap, and it is")
print("why the chapter's batch-norm run does not reach what proper initialisation reaches --")
print("though this measurement alone does not quantify that difference in accuracy.")
MEASUREMENTS.append(dict(measurement="batch-norm backward: straight-through vs true",
                         max_rel=float(rel), cosine=cos))

straight-through dz vs the true batch-norm dz
  max relative difference 0.734
  cosine similarity       0.9722   (1.0 would mean the same direction)
The chapter states this approximation in Steps 5 and 6. It is a real gap, and it is
why the chapter's batch-norm run does not reach what proper initialisation reaches --
though this measurement alone does not quantify that difference in accuracy.


In [7]:
import json
n_pass = sum(1 for r in RESULTS if r["passed"])
print(f"\n{n_pass} of {len(RESULTS)} ASSERTED checks passed")
if MEASUREMENTS:
    print(f"{len(MEASUREMENTS)} reported measurement(s), not asserted:")
    for m in MEASUREMENTS: print("   ", m)
print(json.dumps(dict(checks=RESULTS, measurements=MEASUREMENTS), indent=1))
assert n_pass == len(RESULTS), "a gradient check failed"



10 of 10 ASSERTED checks passed
1 reported measurement(s), not asserted:
    {'measurement': 'batch-norm backward: straight-through vs true', 'max_rel': 0.7342219783059878, 'cosine': 0.9721868970590041}
{
 "checks": [
  {
   "check": "one step of train(): W[0] after the update",
   "max_abs": 4.951594689828198e-14,
   "max_rel": 7.276468634597267e-14,
   "scale": 0.6804941982825239,
   "tol": 1e-09,
   "passed": true
  },
  {
   "check": "one step of train(): b[0] after the update",
   "max_abs": 5.739462724529965e-14,
   "max_rel": 2.2682515567905157e-11,
   "scale": 0.0025303466484339473,
   "tol": 1e-09,
   "passed": true
  },
  {
   "check": "one step of train(): W[1] after the update",
   "max_abs": 1.149080830487037e-13,
   "max_rel": 1.282295241695236e-13,
   "scale": 0.8961125278511638,
   "tol": 1e-09,
   "passed": true
  },
  {
   "check": "one step of train(): b[1] after the update",
   "max_abs": 7.648309069407944e-14,
   "max_rel": 2.0795291914633676e-11,
   "scale": 0.00